In [1]:
import pandas as pd
import re
import os
import sys
sys.path.append("stable-baselines3")
sys.path.append("..")
sys.path.append("强化学习")
import gymnasium as gym
from 强化学习 import load_datas,LivestockEnvConfig
from gymnasium.envs.registration import register
# from livestockEnvV2 import load_datas,LivestockEnvConfig

register(
    id='LivestockEnv-v2',
    entry_point='livestockEnvV2:LivestockEnv',
)
version = 'aaa'
country = 'aus'
FileName = '澳大利亚空间优化更新PB第一步.xlsx'

In [2]:
df = pd.read_excel(rf'../results/{version}/{country}/PPO.xlsx', index_col=0)

ID_move_in, \
ID_move_out, \
Move_out,\
Move_in,\
N_demand_move_in,\
N_demand_Coef_move_in,\
Ammonia_move_in,\
Ammonia_Coef_move_in,\
sensitivity_move_in,\
relative_pm25_move_in,\
N_demand_move_out,\
N_demand_Coef_move_out,\
Ammonia_move_out,\
Ammonia_Coef_move_out,\
sensitivity_move_out,\
relative_pm25_move_out= load_datas(country, FileName, province=None)
if not os.path.exists(f"../results/{version}/{country}/"):
    os.makedirs(f"../results/{country}/")

In [3]:

config = LivestockEnvConfig(country, 
                            Reward_priority=[4, 2, 1], 
                            thresholds=[0, 0], 
                            mobility_ratio=0.1,
                            max_steps=40000,
                            df_path=FileName)
env = gym.make('LivestockEnv-v2', config=config)  


### 合并移动方案

In [4]:
import torch
import re

def string_to_tensor(string):
    # 使用正则表达式提取数字部分
    numbers = re.findall(r'\d+', string)
    # 将数字转换为列表
    numbers = [int(num) for num in numbers]
    # 创建 tensor
    tensor = torch.tensor(numbers)
    return tensor


In [5]:
df["amount"] = df["amount"].apply(lambda x : string_to_tensor(x))

df[Move_out.columns] = 0

for i in range(len(df)):
    for a in range(env.action_len):
        df.iloc[i, a-env.action_len] = df.loc[i, "amount"][a].item()

df.drop(["amount", "action_mask.sum()"], axis=1, inplace=True)
merged_data = df.groupby(list(df.columns[:8])).agg(sum).reset_index()
merged_data.to_excel(f'../results/{version}/{country}/PPO_concated3.xlsx', index=False)
merged_data

/home/yanisy/.local/lib/python3.11/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.action_len to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.action_len` for environment variables or `env.get_wrapper_attr('action_len')` that will search the reminding wrappers.
  logger.warn(
/tmp/ipykernel_3708473/2848592821.py:10: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  merged_data = df.groupby(list(df.columns[:8])).agg(sum).reset_index()


,move_out_idx,move_in_idx,ID_move_out,city_move_out,county_move_out,ID_move_in,city_move_in,county_move_in,reward,num Dairy cattle(头）,num Meat cattle(头）,num Sheep(头）,num Pigs(头）,num County broiler(只）,num County Layers(只）
0,0,119,101021007,101,Braidwood,204031069,Victoria,Bright - Mount Beauty,-0.000970,0,0,0,0,0,97
1,0,122,101021007,101,Braidwood,205021084,Victoria,Lakes Entrance,2.201926,0,6630,0,0,0,65
2,0,170,101021007,101,Braidwood,215031403,Victoria,Robinvale,21.694095,0,0,50652,0,0,0
3,0,419,101021007,101,Braidwood,701031033,101,Koolpinyah,6.295197,0,22873,0,0,0,0
4,1,25,101021611,101,Queanbeyan Surrounds,105011095,101,Nyngan - Warren,9.964331,0,8870,94601,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1119,509,178,604031095,101,Smithton,216021412,Victoria,Moira,5.977683,8644,4831,135,0,0,0
1120,510,287,702011054,101,Yuendumu - Anmatjere,407031161,South Australia,Karoonda - Lameroo,15.970264,0,21306,0,0,0,0
1121,511,11,702051068,101,Victoria River,103021065,101,Forbes,2.956819,0,14340,0,5,0,0
1122,511,309,702051068,101,Victoria River,509021238,Western Australia,Dowerin,9.925439,0,231429,0,4,0,0


### 生成移动后移出城市剩余数量

In [6]:
import torch
merged_data = pd.read_excel(f'../results/{version}/{country}/PPO_concated3.xlsx')
Move_out_copy = Move_out.copy()
N_demand_move_out_copy = N_demand_move_out.copy()
Ammonia_move_out_copy = Ammonia_move_out.copy()

In [7]:
Move_out_tensor_Coef_ammonia = torch.tensor(Ammonia_Coef_move_out.values)
Move_out_tensor_Coef_N_demand = torch.tensor(N_demand_Coef_move_out.values)

In [8]:
for idx, line in merged_data.iterrows():
    out_idx = int(line["move_out_idx"])
    Move_out_copy.iloc[out_idx, :] -= line.iloc[-env.action_len:]
    amounts = torch.tensor(line.iloc[-env.action_len:].values.astype(int))
    N_demand_move_out_copy.iloc[out_idx] -= (amounts.double() @ Move_out_tensor_Coef_N_demand[out_idx, :]).item()
    Ammonia_move_out_copy.iloc[out_idx] -= (amounts.double() @ Move_out_tensor_Coef_ammonia[out_idx, :]).item()
    # livestock_PB_Move_out_copy[out_idx] -= (amounts.double() @ Move_out_tensor_Coef_livestock_PB[out_idx, :]).item()
pd.concat([ID_move_out,Move_out_copy,N_demand_move_out_copy,Ammonia_move_out_copy], axis=1).to_excel(f"../results/{version}/{country}/move_out_result.xlsx", index=False)
# pd.concat([ID_move_out,Move_out_copy], axis=1).to_excel(f"../results/{country}/move_out_result1.xlsx", index=False)


/home/yanisy/.local/lib/python3.11/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.action_len to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.action_len` for environment variables or `env.get_wrapper_attr('action_len')` that will search the reminding wrappers.
  logger.warn(


### 生成移动后移入城市剩余数量

In [9]:
N_demand_Move_in_copy = N_demand_move_in.copy()
Move_in_tensor_Coef_N_demand = torch.tensor(N_demand_Coef_move_in.values)
Ammonia_Move_in_copy = Ammonia_move_in.copy()
Move_in_tensor_Coef_ammonia = torch.tensor(Ammonia_Coef_move_in.values)
Move_in_copy = Move_in.copy()

In [10]:
for idx, line in merged_data.iterrows():
    out_idx = int(line["move_out_idx"])
    in_idx = int(line["move_in_idx"])
    Move_in_copy.iloc[in_idx, :] += line.iloc[-env.action_len:]
    amounts = torch.tensor(line. iloc[-env.action_len:].values.astype(int))
    N_demand_Move_in_copy.iloc[in_idx] += (amounts.double() @ Move_in_tensor_Coef_N_demand[in_idx, :]).item()
    Ammonia_Move_in_copy.iloc[in_idx] -= (amounts.double() @ Move_in_tensor_Coef_ammonia[in_idx, :]).item()
pd.concat([ID_move_in,Move_in_copy,N_demand_Move_in_copy,Ammonia_Move_in_copy], axis=1).to_excel(f"../results/{version}/{country}/move_in_result3.xlsx", index=False)
# Move_in_copy[[*Move_in_copy.columns[:env.k],"氨排放差距","承载力差距",*Move_out.columns[env.k:]]].to_excel(f"../results/{country}/v4/move_in_result1.xlsx", index=False)


/home/yanisy/.local/lib/python3.11/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.action_len to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.action_len` for environment variables or `env.get_wrapper_attr('action_len')` that will search the reminding wrappers.
  logger.warn(


In [11]:
# pd.merge(df_out.loc[:, move_out_result.columns], move_out_result, on=list(move_out_result.columns[:3]), how='left').to_excel(f"../results/{version}/{country}_省内/move_out_result_concat.xlsx", index=False)
# pd.merge(df_in.loc[:, move_in_result.columns], move_in_result, on=list(move_in_result.columns[:3]), how='left').to_excel(f"../results/{version}/{country}_省内/move_in_result_concat.xlsx", index=False)
